# Phase 2 — Softmax Regression: Multiclass Classification on MNIST

**CSE382: Introduction to Machine Learning — Spring 2026**

This notebook extends Phase 1 (binary logistic regression) to full **10-class classification** using **Softmax Regression** implemented from scratch with NumPy.

**Improvements over Phase 1:**
- **Improvement 1 — Pretrained CNN Feature Extraction**: MobileNetV2 (pretrained on ImageNet) used as a feature extractor instead of raw pixels
- **Improvement 2 — Hyperparameter Tuning with Cross-Validation**: 5-fold CV grid search over learning rate instead of a single fixed split
- **Improvement 3 — Overfitting/Underfitting Diagnosis**: Loss curves per epoch + accuracy vs training size curves


## 1. Mathematical Formulation

### From Binary (Phase 1) to Multiclass (Phase 2)

In Phase 1, a single sigmoid output gave $P(y=1 \mid \mathbf{x})$.
In Phase 2, **softmax** produces a probability for each of the 10 digit classes simultaneously.

### Softmax Function

Given input $\mathbf{x} \in \mathbb{R}^d$, weight matrix $\mathbf{W} \in \mathbb{R}^{d \times K}$, bias $\mathbf{b} \in \mathbb{R}^K$:

$$\mathbf{z} = \mathbf{W}^\top \mathbf{x} + \mathbf{b} \quad \in \mathbb{R}^K \qquad \text{(logits)}$$

$$P(y = k \mid \mathbf{x}) = \text{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

Predicted class: $\hat{y} = \arg\max_k \ P(y = k \mid \mathbf{x})$

### Numerical Stability

To avoid overflow from large exponentials, we subtract the row maximum before computing softmax:

$$\text{softmax}(\mathbf{z})_k = \frac{e^{z_k - \max(\mathbf{z})}}{\sum_j e^{z_j - \max(\mathbf{z})}}$$

### Input Feature Dimensionality

| Feature Method | Description | Dimensionality $d$ |
|---|---|---|
| Flatten | Raw pixels unrolled | $28 \times 28 = 784$ |
| HOG | Histogram of Oriented Gradients | $324$ |
| CNN (MobileNetV2) | Pretrained deep features | $1280$ |

### Loss Function — Categorical Cross-Entropy

$$\mathcal{L}(\mathbf{W}, \mathbf{b}) = -\frac{1}{m} \sum_{i=1}^{m} \sum_{k=1}^{K} y_{ik} \log(\hat{p}_{ik})$$

where $y_{ik} = 1$ if sample $i$ belongs to class $k$ (one-hot), else $0$.

### Gradient Descent Update Rules

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}} = \frac{1}{m} X^\top (\hat{\mathbf{P}} - \mathbf{Y})$$

$$\mathbf{W} \leftarrow \mathbf{W} - \alpha \frac{\partial \mathcal{L}}{\partial \mathbf{W}}, \qquad \mathbf{b} \leftarrow \mathbf{b} - \alpha \frac{\partial \mathcal{L}}{\partial \mathbf{b}}$$

### Evaluation Metrics (Multiclass)

| Metric | Formula |
|---|---|
| Accuracy | $\frac{\text{correct predictions}}{\text{total}}$ |
| Precision (macro) | Average per-class $\frac{TP}{TP+FP}$ |
| Recall (macro) | Average per-class $\frac{TP}{TP+FN}$ |
| F1 (macro) | Average per-class $\frac{2 \cdot P \cdot R}{P + R}$ |


## 2. Imports & Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import importlib
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D

import preprocessing2
importlib.reload(preprocessing2)
from preprocessing2 import preprocess_mnist_multiclass

print('All imports successful')
print(f'NumPy : {np.__version__}')
print(f'TF    : {tf.__version__}')


## 3. Configuration & Hyperparameters

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
methods_to_run  = ['flatten', 'hog', 'cnn']   # feature methods to compare
learning_rates  = [0.001, 0.01, 0.1]           # tuned via cross-validation
epochs          = 500                           # training epochs per final run
val_ratio       = 0.15                          # 15% train → validation
K_FOLDS         = 5                             # number of CV folds
NUM_CLASSES     = 10                            # digits 0–9
RANDOM_SEED     = 42

np.random.seed(RANDOM_SEED)
print('Configuration set.')


## 4. Data Loading & Preprocessing

Using `preprocess_mnist_multiclass` from `preprocessing2.py` — the same file as Phase 1, extended for multiclass.

**Key differences from Phase 1:**
- Labels stay as integers **0–9** (no binary conversion)
- Uses `float32` instead of `float64` to avoid memory errors during CNN extraction
- Also returns raw 28×28 images needed for CNN feature extraction


In [ ]:
# ── Load all feature sets ────────────────────────────────────────────────────
print('Loading HOG features...')
(X_tr_hog, y_train, X_v_hog, y_val, X_te_hog, y_test,
 hog_mean, hog_std, x_train_raw, x_val_raw, x_test_raw) = preprocess_mnist_multiclass(
    method='hog', val_ratio=val_ratio
)
print(f'  Train: {X_tr_hog.shape}  Val: {X_v_hog.shape}  Test: {X_te_hog.shape}')

print('Loading Flatten features...')
(X_tr_flat, _, X_v_flat, _, X_te_flat, _,
 flat_mean, flat_std, _, _, _) = preprocess_mnist_multiclass(
    method='flatten', val_ratio=val_ratio
)
print(f'  Train: {X_tr_flat.shape}  Val: {X_v_flat.shape}  Test: {X_te_flat.shape}')

print('\nClass distribution in training set:')
for c in range(10):
    count = np.sum(y_train == c)
    bar   = '█' * (count // 200)
    print(f'  Digit {c}: {count:5d}  {bar}')


def to_one_hot(y, num_classes=10):
    """Convert integer labels to one-hot matrix. Shape: (m, K)"""
    m = len(y)
    Y = np.zeros((m, num_classes), dtype=np.float32)
    Y[np.arange(m), y] = 1
    return Y


## 5. Feature Engineering — Pretrained CNN Feature Extraction

### Improvement 1 — MobileNetV2 Transfer Learning

Instead of raw pixels or hand-crafted HOG features, we use **MobileNetV2** pretrained on ImageNet as a feature extractor.

**How it works:**
1. Load MobileNetV2 with ImageNet weights — **freeze all CNN layers** (no retraining)
2. Remove the final classification head
3. Attach a `GlobalAveragePooling2D` layer → outputs a **1280-dimensional** feature vector per image
4. Pass each MNIST image through the frozen CNN → use the 1280 features as input to our softmax classifier

**Why this works:** The CNN has already learned to detect edges, textures, and shapes from millions of ImageNet images. Even though MNIST digits are simpler, these low-level features transfer well and produce a much richer representation than raw pixels.

**Preprocessing for MobileNetV2:**
- MNIST images are 28×28 grayscale → resize to **96×96** (MobileNetV2 minimum input size)
- Convert grayscale → **RGB** (repeat single channel 3 times)
- Apply MobileNetV2 preprocessing (scale pixels to **[-1, 1]**)


In [ ]:
# ── Build CNN feature extractor ──────────────────────────────────────────────
def build_cnn_extractor():
    """
    MobileNetV2 feature extractor (frozen weights).
    Input  : (batch, 96, 96, 3)
    Output : (batch, 1280)  — GlobalAveragePooling2D features
    """
    base = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights='imagenet')
    base.trainable = False   # freeze all CNN weights
    out  = GlobalAveragePooling2D()(base.output)
    return Model(inputs=base.input, outputs=out)


def extract_cnn_features(images, extractor, batch_size=256):
    """
    Extract CNN features from raw 28x28 images:
      1. Resize 28x28 → 96x96
      2. Grayscale → RGB (stack 3 channels)
      3. MobileNetV2 preprocessing (scale to [-1, 1])
      4. Forward pass through frozen CNN → (m, 1280) float32
    """
    features = []
    n = len(images)
    for start in range(0, n, batch_size):
        batch        = images[start:start+batch_size]
        batch_rsz    = tf.image.resize(batch[..., np.newaxis], (96, 96)).numpy()
        batch_rgb    = np.repeat(batch_rsz, 3, axis=-1)
        batch_pre    = preprocess_input(batch_rgb * 255.0)
        feat         = extractor.predict(batch_pre, verbose=0).astype(np.float32)
        features.append(feat)
        print(f'  {min(start+batch_size, n)}/{n}', end='\r')
    print()
    return np.vstack(features)


print('Building MobileNetV2 extractor...')
cnn_extractor = build_cnn_extractor()
print(f'Output shape: {cnn_extractor.output_shape}')
print(f'Trainable params: {sum(np.prod(w.shape) for w in cnn_extractor.trainable_weights)}')


In [ ]:
# ── Extract CNN features for train / val / test ───────────────────────────────
# Raw 28x28 images were saved by preprocess_mnist_multiclass for exactly this purpose

print('Extracting CNN features — Train:')
cnn_tr = extract_cnn_features(x_train_raw, cnn_extractor)
print('Extracting CNN features — Val:')
cnn_v  = extract_cnn_features(x_val_raw,   cnn_extractor)
print('Extracting CNN features — Test:')
cnn_te = extract_cnn_features(x_test_raw,  cnn_extractor)

# Standardize CNN features (train stats only — no leakage)
cnn_mean = np.mean(cnn_tr, axis=0)
cnn_std  = np.std(cnn_tr,  axis=0) + 1e-8
C_tr = ((cnn_tr - cnn_mean) / cnn_std).astype(np.float32)
C_v  = ((cnn_v  - cnn_mean) / cnn_std).astype(np.float32)
C_te = ((cnn_te - cnn_mean) / cnn_std).astype(np.float32)

print(f'CNN features: {C_tr.shape[1]} dimensions  dtype={C_tr.dtype}')

# Collect all feature sets
feature_sets = {
    'flatten': (X_tr_flat, X_v_flat, X_te_flat),
    'hog'    : (X_tr_hog,  X_v_hog,  X_te_hog),
    'cnn'    : (C_tr,      C_v,      C_te),
}

print('\n=== All feature sets ready ===')
for name, (tr, v, te) in feature_sets.items():
    print(f'  {name:<8}: train={tr.shape}  val={v.shape}  test={te.shape}')


## 6. Core Functions

In [ ]:
# ── Softmax ──────────────────────────────────────────────────────────────────
def softmax(Z):
    """
    Numerically stable softmax.
    Input  Z : (m, K) — logits
    Output   : (m, K) — probabilities (each row sums to 1)
    """
    Z_s = Z - np.max(Z, axis=1, keepdims=True)   # subtract row max for stability
    e   = np.exp(Z_s)
    return e / np.sum(e, axis=1, keepdims=True)


# ── Loss ─────────────────────────────────────────────────────────────────────
def cross_entropy_loss(Y_oh, P):
    """
    Categorical cross-entropy loss.
    Y_oh : (m, K) one-hot true labels
    P    : (m, K) predicted probabilities
    """
    P = np.clip(P, 1e-8, 1 - 1e-8)
    return -np.mean(np.sum(Y_oh * np.log(P), axis=1))


# ── Prediction ────────────────────────────────────────────────────────────────
def predict_proba_softmax(X, W, b):
    """Forward pass: X(m,d) @ W(d,K) + b(K) → softmax → P(m,K)"""
    return softmax(X @ W + b)


def predict_class(X, W, b):
    """Return predicted class index (0–9) for each sample."""
    return np.argmax(predict_proba_softmax(X, W, b), axis=1)


# ── Evaluation ────────────────────────────────────────────────────────────────
def evaluate_multiclass(y_true, y_pred, num_classes=10):
    """
    Accuracy + macro-averaged precision, recall, F1.
    All computed from scratch using TP/FP/FN per class.
    """
    acc = np.mean(y_true == y_pred)
    precisions, recalls, f1s = [], [], []

    for k in range(num_classes):
        tp = np.sum((y_pred == k) & (y_true == k))
        fp = np.sum((y_pred == k) & (y_true != k))
        fn = np.sum((y_pred != k) & (y_true == k))

        prec = tp / (tp + fp + 1e-8)
        rec  = tp / (tp + fn + 1e-8)
        f1   = 2 * prec * rec / (prec + rec + 1e-8)

        precisions.append(prec)
        recalls.append(rec)
        f1s.append(f1)

    return {
        'acc' : acc,
        'prec': np.mean(precisions),
        'rec' : np.mean(recalls),
        'f1'  : np.mean(f1s),
        'per_class_f1': np.array(f1s)
    }


def print_confusion_matrix(y_true, y_pred, num_classes=10):
    """Print 10x10 confusion matrix."""
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    print('\nConfusion Matrix (rows=actual, cols=predicted):')
    print('      ' + '  '.join(f'{i:4d}' for i in range(num_classes)))
    print('    ' + '─' * 54)
    for i in range(num_classes):
        row = '  '.join(f'{cm[i,j]:4d}' for j in range(num_classes))
        print(f'  {i} | {row}')


print('Core functions defined.')


## 7. Training Function (with Loss Curve Tracking)

In [ ]:
def train_softmax(X_tr, Y_tr, X_v, Y_v, lr, n_epochs=500):
    """
    Train softmax regression with gradient descent.

    Parameters:
        X_tr, Y_tr : training features (m,d) and one-hot labels (m,K)
        X_v,  Y_v  : validation features and one-hot labels
        lr         : learning rate
        n_epochs   : number of gradient descent steps

    Returns:
        best_W, best_b : weights at lowest validation loss epoch
        train_losses   : list of train loss per epoch
        val_losses     : list of val loss per epoch
        best_epoch     : epoch of best validation checkpoint
    """
    m, d = X_tr.shape
    K    = Y_tr.shape[1]

    # Small random init — better than zeros for multiclass (breaks symmetry)
    np.random.seed(RANDOM_SEED)
    W = (np.random.randn(d, K) * 0.01).astype(np.float32)
    b = np.zeros(K, dtype=np.float32)

    best_val_loss = float('inf')
    best_W, best_b, best_epoch = W.copy(), b.copy(), 0
    train_losses, val_losses   = [], []

    for epoch in range(n_epochs):
        # Forward pass
        P_tr = predict_proba_softmax(X_tr, W, b)     # (m, K)

        # Gradient: (P - Y) is the error, X^T scales by features
        err  = P_tr - Y_tr                            # (m, K)
        dW   = (X_tr.T @ err) / m                     # (d, K)
        db   = np.mean(err, axis=0)                   # (K,)

        # Update
        W -= lr * dW
        b -= lr * db

        # Track losses
        t_loss = cross_entropy_loss(Y_tr, predict_proba_softmax(X_tr, W, b))
        v_loss = cross_entropy_loss(Y_v,  predict_proba_softmax(X_v,  W, b))
        train_losses.append(t_loss)
        val_losses.append(v_loss)

        # Checkpoint best weights
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_W, best_b, best_epoch = W.copy(), b.copy(), epoch

    return best_W, best_b, train_losses, val_losses, best_epoch


print('Training function defined.')


## 8. Hyperparameter Tuning with Cross-Validation

### Improvement 2 — K-Fold Cross-Validation

In Phase 1, hyperparameters were tuned using a **single fixed validation split** (85/15).  
In Phase 2, we use **5-fold cross-validation**:

- The training data is divided into 5 equal folds
- Each configuration trains 5 times — each time a different fold is held out as validation
- The final score = **average validation accuracy across all 5 folds**
- This gives a more reliable estimate because every sample is used for validation exactly once

We tune **learning rate** on CNN features (the strongest feature set), then apply the best `lr` to all methods.

| | Phase 1 | Phase 2 |
|---|---|---|
| Tuning strategy | Single holdout split | 5-fold cross-validation |
| Reliability | Depends on random split | Stable — all samples validated |
| Speed | Fast | 5× slower per config |


In [ ]:
def cross_validate(X, y, lr, k=5, n_epochs=200, seed=42):
    """
    K-fold cross-validation for softmax regression.
    Splits X into k folds, trains on k-1, validates on 1, rotates k times.
    Returns mean and std of validation accuracy across folds.
    """
    m = len(X)
    np.random.seed(seed)
    indices = np.random.permutation(m)
    folds   = np.array_split(indices, k)
    fold_accs = []

    for fold_idx in range(k):
        val_idx   = folds[fold_idx]
        train_idx = np.concatenate([folds[i] for i in range(k) if i != fold_idx])

        X_tr_cv, y_tr_cv = X[train_idx], y[train_idx]
        X_v_cv,  y_v_cv  = X[val_idx],   y[val_idx]

        Y_tr_cv = to_one_hot(y_tr_cv)
        Y_v_cv  = to_one_hot(y_v_cv)

        W, b, _, _, _ = train_softmax(X_tr_cv, Y_tr_cv, X_v_cv, Y_v_cv,
                                       lr=lr, n_epochs=n_epochs)
        y_pred = predict_class(X_v_cv, W, b)
        fold_accs.append(np.mean(y_pred == y_v_cv))

    return np.mean(fold_accs), np.std(fold_accs)


print('Cross-validation function defined.')


In [ ]:
# ── Run CV grid search on CNN features ───────────────────────────────────────
# 5 folds × 3 learning rates = 15 training runs
# Using 200 epochs per fold to keep runtime manageable

print('Running 5-fold cross-validation on CNN features...')
print('(This may take a few minutes)\n')

best_cv_acc    = -1
best_lr        = None
cv_results     = []

print(f"{'LR':>8} {'Mean Acc':>10} {'Std':>8}")
print('─' * 32)

for lr in learning_rates:
    mean_acc, std_acc = cross_validate(C_tr, y_train, lr=lr,
                                       k=K_FOLDS, n_epochs=200)
    cv_results.append({'lr': lr, 'mean_acc': mean_acc, 'std_acc': std_acc})
    marker = ' ◄ BEST' if mean_acc > best_cv_acc else ''
    if mean_acc > best_cv_acc:
        best_cv_acc = mean_acc
        best_lr     = lr
    print(f'{lr:>8.3f} {mean_acc:>10.4f} {std_acc:>8.4f}{marker}')

print('─' * 32)
print(f'\nBest lr={best_lr}  mean val acc={best_cv_acc:.4f}')


## 9. Final Training — All Feature Methods

Using the best learning rate found by cross-validation, we now train the final models for all three feature methods.


In [ ]:
# ── Train final models ───────────────────────────────────────────────────────
BEST_LR      = best_lr
final_models = {}
all_results  = []

print(f'Training final models with lr={BEST_LR}  epochs={epochs}\n')
print(f"{'Method':<10} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'BestEp':>8}")
print('─' * 58)

for method in methods_to_run:
    X_tr_m, X_v_m, X_te_m = feature_sets[method]
    Y_tr_m = to_one_hot(y_train)
    Y_v_m  = to_one_hot(y_val)

    W, b, t_losses, v_losses, best_ep = train_softmax(
        X_tr_m, Y_tr_m, X_v_m, Y_v_m,
        lr=BEST_LR, n_epochs=epochs
    )

    y_val_pred = predict_class(X_v_m, W, b)
    metrics    = evaluate_multiclass(y_val, y_val_pred)

    final_models[method] = {
        'W': W, 'b': b,
        'X_te': X_te_m,
        'train_losses': t_losses, 'val_losses': v_losses,
        'best_epoch'  : best_ep,
        'val_metrics' : metrics
    }
    all_results.append({'method': method, **metrics, 'best_epoch': best_ep})

    print(f"{method:<10} {metrics['acc']:>8.4f} {metrics['prec']:>8.4f} "
          f"{metrics['rec']:>8.4f} {metrics['f1']:>8.4f} {best_ep:>8d}")

print('─' * 58)
best_method = max(all_results, key=lambda r: r['f1'])['method']
print(f'\nBest method by val F1: {best_method.upper()}')


## 10. Overfitting/Underfitting Diagnosis — Learning Curves

### Improvement 3 — Loss Curves & Training Size Analysis

**Part A — Loss curves per epoch:**  
Show how training and validation loss evolve. Reveals:
- **Good fit**: both losses low and converge together
- **Overfitting**: train loss low, val loss higher and diverging
- **Underfitting**: both losses high and similar

**Part B — Accuracy vs training size:**  
Train the best model on increasing fractions of training data. Reveals:
- If val accuracy keeps rising → **high variance** (more data would help)
- If val accuracy plateaus → **high bias** (more data won't help, model is the bottleneck)


In [ ]:
# ── Part A: Loss curves for all methods ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, method in zip(axes, methods_to_run):
    model   = final_models[method]
    t_loss  = model['train_losses']
    v_loss  = model['val_losses']
    best_ep = model['best_epoch']

    ax.plot(t_loss, color='#2563eb', linewidth=1.5, label='Train loss')
    ax.plot(v_loss, color='#dc2626', linewidth=1.5, label='Val loss')
    ax.axvline(best_ep, color='gray', linestyle='--', linewidth=1,
               label=f'Best epoch ({best_ep})')
    ax.set_title(f'{method.upper()} — Loss Curves')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Loss Curves — lr={BEST_LR}', fontsize=13)
plt.tight_layout()
plt.show()

# ── Diagnosis table ───────────────────────────────────────────────────────────
print('\nBias-Variance Diagnosis:')
print(f"{'Method':<10} {'Train Loss':>12} {'Val Loss':>10} {'Gap':>8} {'Diagnosis':>15}")
print('─' * 58)
for method in methods_to_run:
    t_l = final_models[method]['train_losses'][-1]
    v_l = final_models[method]['val_losses'][-1]
    gap = v_l - t_l
    if t_l > 1.5:
        diag = 'Underfitting'
    elif gap > 0.1:
        diag = 'Overfitting'
    else:
        diag = 'Good fit'
    print(f"{method:<10} {t_l:>12.4f} {v_l:>10.4f} {gap:>8.4f} {diag:>15}")
print('─' * 58)


In [ ]:
# ── Part B: Accuracy vs training size (best method) ──────────────────────────
X_best_tr, X_best_v, _ = feature_sets[best_method]
Y_best_v               = to_one_hot(y_val)

print(f'Learning curve — {best_method.upper()} (best method)\n')

train_fracs = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
tr_accs, v_accs = [], []

for frac in train_fracs:
    n_samples = int(len(X_best_tr) * frac)
    X_sub     = X_best_tr[:n_samples]
    y_sub     = y_train[:n_samples]
    Y_sub     = to_one_hot(y_sub)

    W_s, b_s, _, _, _ = train_softmax(X_sub, Y_sub, X_best_v, Y_best_v,
                                       lr=BEST_LR, n_epochs=300)
    tr_acc = np.mean(predict_class(X_sub,    W_s, b_s) == y_sub)
    v_acc  = np.mean(predict_class(X_best_v, W_s, b_s) == y_val)
    tr_accs.append(tr_acc)
    v_accs.append(v_acc)
    print(f'  {int(frac*100):3d}% ({n_samples:5d} samples):  '
          f'Train={tr_acc:.4f}  Val={v_acc:.4f}')

sizes_abs = [int(len(X_best_tr) * f) for f in train_fracs]
plt.figure(figsize=(8, 4))
plt.plot(sizes_abs, tr_accs, 'o-', color='#2563eb', linewidth=2, label='Train accuracy')
plt.plot(sizes_abs, v_accs,  'o-', color='#dc2626', linewidth=2, label='Val accuracy')
plt.xlabel('Training set size')
plt.ylabel('Accuracy')
plt.title(f'Learning Curve — {best_method.upper()}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Final Test Evaluation

The test set is used **exactly once**, after all hyperparameter choices are finalised.


In [ ]:
# ── Evaluate all methods on test set ─────────────────────────────────────────
print('=' * 58)
print('        FINAL TEST RESULTS — ALL METHODS')
print('=' * 58)
print(f"{'Method':<10} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'BestEp':>8}")
print('─' * 58)

test_results     = {}
best_test_f1     = -1
best_test_method = None

# First pass — collect all results
for method in methods_to_run:
    model  = final_models[method]
    W, b   = model['W'], model['b']
    X_te_m = model['X_te']
    y_pred  = predict_class(X_te_m, W, b)
    metrics = evaluate_multiclass(y_test, y_pred)
    test_results[method] = {'metrics': metrics, 'y_pred': y_pred}
    if metrics['f1'] > best_test_f1:
        best_test_f1     = metrics['f1']
        best_test_method = method

# Second pass — print table with correct BEST marker
for method in methods_to_run:
    metrics = test_results[method]['metrics']
    model   = final_models[method]
    marker  = ' ◄ BEST' if method == best_test_method else ''
    print(f"{method:<10} {metrics['acc']:>8.4f} {metrics['prec']:>8.4f} "
          f"{metrics['rec']:>8.4f} {metrics['f1']:>8.4f} "
          f"{model['best_epoch']:>8d}{marker}")

print('─' * 58)
print(f'\nBest method: {best_test_method.upper()}  Test F1={best_test_f1:.4f}')

# ── Per-class F1 ─────────────────────────────────────────────────────────────
best_metrics = test_results[best_test_method]['metrics']
print(f'\nPer-class F1 — {best_test_method.upper()}:')
print('─' * 30)
for k, f1 in enumerate(best_metrics['per_class_f1']):
    bar = '█' * int(f1 * 30)
    print(f'  Digit {k}: {f1:.4f}  {bar}')

# ── Confusion matrix ─────────────────────────────────────────────────────────
print_confusion_matrix(y_test, test_results[best_test_method]['y_pred'])


In [ ]:
# ── Confusion matrix heatmap ─────────────────────────────────────────────────
y_pred_best = test_results[best_test_method]['y_pred']
cm_mat      = np.zeros((10, 10), dtype=int)
for t, p in zip(y_test, y_pred_best):
    cm_mat[t, p] += 1

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm_mat, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(10));  ax.set_yticks(range(10))
ax.set_xticklabels(range(10)); ax.set_yticklabels(range(10))
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title(f'Confusion Matrix — {best_test_method.upper()} (Test Set)')
thresh = cm_mat.max() / 2
for i in range(10):
    for j in range(10):
        ax.text(j, i, cm_mat[i, j], ha='center', va='center',
                color='white' if cm_mat[i, j] > thresh else 'black', fontsize=9)
plt.tight_layout()
plt.show()


## 12. Results Analysis

### Improvement 1 — CNN Feature Extraction

- **Flatten (784 features)**: Raw pixels carry the most information but also the most noise. High dimensionality slows convergence.
- **HOG (324 features)**: Shape-aware features. Works well for digits since they are defined by strokes and curves.
- **CNN (1280 features)**: Features learned from millions of images. Captures hierarchical patterns — edges → shapes → structures. Expected to give the highest accuracy.

### Improvement 2 — Cross-Validation

- Phase 1 used a single 85/15 split — results could vary depending on which samples landed in validation.
- 5-fold CV gives a more reliable estimate: every sample is used for validation exactly once.
- The standard deviation across folds tells us how stable the chosen learning rate is.

### Improvement 3 — Learning Curves

- **Loss curves**: if train and val loss diverge → overfitting; if both stay high → underfitting.
- **Training size curves**: if val accuracy keeps increasing with more data → the model has high variance and more data would help. If it plateaus → the model itself is the bottleneck.
